# Lab 11 · MCP: serve tools through a standard protocol

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. This lab needs no model key.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit', 'mcp']):
    %pip install -q --upgrade "agentic-ai-coursekit[mcp] @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()

## Warm-up · Discover and call one MCP tool

An MCP server exposes a greeting tool. A client discovers its name and calls it through the actual local MCP SDK. This warm-up does not use a model or the network.

In [ ]:
from mcp.server import MCPServer
from mcp import Client

warm_server = MCPServer("greeting")
@warm_server.tool(description="Greet a person by name.")
def greet(name: str) -> str:
    return f"Hello, {name}!"

async with Client(warm_server) as warm_client:
    tools = await warm_client.list_tools()
    print("Discovered tools:")
    for tool in tools.tools:
        print(tool.name)
    result = await warm_client.call_tool("greet", {"name": "Sam"})
    print("Result:", result.content[0].text)

**Next:** Apply the same discover-and-call protocol to an Orders owner.

## Give an agent the owning service's tools
**Start with a problem:** Different agents need a way to find and call approved Orders operations without reading private tables.

**In this lab:** Publish tool names and schemas through MCP and call the owner’s tools. MCP alone does not enforce authorization or data ownership.

**New:** MCP publishes tool descriptions and argument schemas so a client can discover and call them. The microservices lesson is **database ownership**: reporting must not read another service's private tables. It asks the owner through an API or consumes published events instead.

Our small Orders tool server exposes an owner-approved read and a validated decision recorder. The Python dictionary stands in for Orders' private storage. MCP is the agent-facing protocol here; it does not replace the business services' APIs or enforce data ownership by itself. No model key, database server, or deployment is needed.

**At a glance**

![Lab 11 at a glance: MCP server (2 registered tools), then list_tools (names + schemas), then get_order (read), then Choose read pattern (plain Python), then save_decision (write), then State check (rejected calls change nothing).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/11.svg)

## Think about the example

Orders owns its private data. Reporting can use an exposed API or published events. A broad search across services may benefit from a CQRS read model, while a small lookup can use API composition.

**Concepts:** Database per service; API composition versus CQRS; Tool schema and validation.

Use the scenario above to consider these questions before running the code:

1. Why should reporting not read Orders tables directly?
2. What requirement makes CQRS preferable to API composition?

## The idea in context

MCP standardizes tool discovery and calls. The lab runs its server locally in process; it does not deploy business microservices.

Database per service means private ownership; public APIs and events expose selected facts.

| Key term | Meaning |
| --- | --- |
| **MCP server** | A program that publishes tools, and other context, through the protocol. |
| **MCP client** | A program that discovers and calls a server's tools. |
| **Input schema** | A machine-readable description of a tool's arguments and their types. |
| **Discovery** | Asking a server what it offers, instead of hard-coding it. |

## Pattern · Registry & Adapter

Discover tools through a stable interface instead of importing each implementation. The trade-off is another protocol boundary to validate and observe. Here the MCP server adapts an Orders read operation and a decision write operation; the client discovers their schemas first.

## Main exercise at a glance

1. Keep the data behind its owner
2. A validated write tool
3. Discover the schemas
4. Read, then record two pattern decisions
5. Reject broken calls without changing state

## Step 1 · Keep the data behind its owner

Only the Orders server owns `ORDERS`. `get_order` returns a copy of the fields this interface exposes; callers do not receive a database connection or a reference they can mutate. This mirrors database-per-service, so Reporting cannot read Order's private tables directly.

The MCP SDK requires a Python function for each tool. The decorator registers the function so a client can discover and call it. The `print` at the end of the next cell calls the plain Python function directly; the MCP path — a client discovering and calling the registered tool over the protocol — starts in Step 3.

In [ ]:
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

ORDERS = {"101": {"status": "PENDING", "total": 42},
          "102": {"status": "APPROVED", "total": 72}}
server = MCPServer("orders-owner")

@server.tool(description="Read an order through its owner's interface, never through shared tables.")
def get_order(order_id: str) -> dict:
    if order_id not in ORDERS:
        raise ToolError(f"Unknown order {order_id!r}")
    return {"order_id": order_id, **ORDERS[order_id]}

print(get_order("101"))

## Step 2 · A validated write tool

The dictionary `decisions` is only a classroom decision log. The tool accepts just the two exercise requirements and the two pattern labels, so a mistaken call returns an error instead of corrupting state. Which label is *right* for a requirement is a design question about the requirements — not something MCP decides.

In [ ]:
LABELS = ("API composition", "CQRS")
decisions = {}

@server.tool(description="Record a proposed read pattern for a named requirement.")
def save_decision(requirement: str, choice: str) -> str:
    if requirement not in ("order details by id", "search customer history"):
        raise ToolError("Unknown requirement")
    if choice not in LABELS:
        raise ToolError(f"choice must be one of {LABELS}")
    decisions[requirement] = choice
    return f"Saved {choice!r} for {requirement}"

## Step 3 · Discover the schemas

The client learns tool names, descriptions, and arguments from the server. It does not inspect `ORDERS`. Colab accepts `await` at the top level; the official MCP SDK handles the local protocol exchange.

`list_tools()` asks the MCP server for registered tool names, descriptions, and input schemas. This is an in-process SDK exchange; no service is deployed.

In [ ]:
from mcp import Client

async with Client(server) as client:
    listing = await client.list_tools()
for tool in listing.tools:
    parts = []
    for name, spec in tool.input_schema["properties"].items():
        parts.append(f"{name}: {spec['type']}")
    args = ", ".join(parts)
    print(f"{tool.name}({args})\n    {tool.description}")

## Step 4 · Read, then record two pattern decisions

These are fixed client calls. A model could choose tools as in Lab 05, but this lab isolates the protocol and the boundary. CQRS also costs another read model and eventual consistency; API composition depends on its source services being available.

The Python code chooses a pattern for each example, then asks the server to save it. The MCP server provides tools here; it is not a model choosing the pattern.

In [ ]:
import json

async with Client(server) as client:
    result = await client.call_tool("get_order", {"order_id": "101"})
    print("Owner returned:", json.loads(result.content[0].text))
    for requirement, choice in [("order details by id", "API composition"), ("search customer history", "CQRS")]:
        saved = await client.call_tool("save_decision", {"requirement": requirement, "choice": choice})
        print(saved.content[0].text)
print("Decision log:", decisions)
assert decisions["search customer history"] == "CQRS"

## Step 5 · Reject broken calls without changing state

A tool error comes back as data. A missing required argument fails schema validation before the tool runs. The last assertion checks the important effect: a rejected request cannot overwrite a valid decision.

In [ ]:
bad_calls = [("get_order", {"order_id": "999"}),
             ("save_decision", {"requirement": "order details by id", "choice": "shared database"}),
             ("save_decision", {"requirement": "order details by id"})]
async with Client(server) as client:
    for tool, arguments in bad_calls:
        result = await client.call_tool(tool, arguments)
        print(f"is_error={result.is_error}: {result.content[0].text.splitlines()[0]}")
        assert result.is_error
assert decisions["order details by id"] == "API composition"

## What you should see

The Orders interface returns order 101 with total 42. Discovery lists two tools. The decision log contains API composition for a small ID lookup and CQRS for history search. Three bad calls return errors; the original decision stays intact.

## Takeaway

MCP gives an agent a discoverable tool boundary. Data ownership and validation remain application responsibilities. A successful protocol call is not proof that a chosen architecture pattern is appropriate.

## Exercise · Read a new order through its owner

1. In Step 1, add order `"103"` to `ORDERS` with `{"status": "PENDING", "total": 55}`. In the `get_order` call in Step 4, change `"order_id": "101"` to `"order_id": "103"`.
2. Rerun Steps 1–4 to rebuild the server, register both tools, and read the new order. Check that `Owner returned:` includes order 103, status `PENDING`, and total 55.
3. Rerun Step 5 to check that broken calls still return errors and keep the valid decision. Explain which component owns the new order and how the client obtains it.

**More to try**

- Explain what boundary a SQL connection for the reporting team would bypass.
- Add `cancel_order(order_id)` as an owner tool that validates the order's state before changing it. Keep the change within Orders.
- In your council, expose a read-only source lookup and a separately authorized decision write. Which data should each tool reveal?